## ETL: Registro de Atendimentos da Defesa Civil do Recife

### Importação das bibliotecas

In [1]:
import pandas as pd

### Extração dos dados

In [2]:
df_2022 = pd.read_csv("data/atendimentos-2022.csv", sep=";", dtype=str, encoding="utf-8")
df_2023 = pd.read_csv("data/atendimentos-2023.csv", sep=";", dtype=str, encoding="utf-8")
df_2024 = pd.read_csv("data/atendimentos-2024.csv", sep=";", dtype=str, encoding="utf-8")

print(df_2022.shape)
print(df_2023.shape)
print(df_2024.shape)

(90425, 16)
(83164, 16)
(80173, 16)


### Transformação dos dados

#### Diagnóstico do problema
Embora as 3 bases tenham as mesmas 16 colunas no cabeçalho, o arquivo de 2022 não possui valores para a coluna **Regional**. Com isso, os dados aparecem deslocados uma coluna para a esquerda: o ano sob "Regional", o mês sob "Ano", e assim por diante. Além disso, o endereço vem quebrado em dois campos, com o nome da rua separado do número anonimizado. A partir da coluna **Bairro**, os dados voltam a ficar alinhados. A comparação abaixo, entre as primeiras linhas de 2022 e de 2023, evidencia o problema.

In [3]:
# exibe as primeiras linhas dos dataframes para verificação
display(df_2022.head(3))
display(df_2023.head(3))

,Regional,Ano,Mês,Data,Ocorrencia,Solicitacao,Endereco,Bairro,Localidade,Grau_de_Risco,Data_da_Acao,Tipo_da_Acao,Quantidade,Altura,Largura,Area_m
0,2022,ABRIL,2022-04-19,Imveis com Danos / Risco,Vistoria em Imveis,"""Rua Jose Moreira Reis","{})!""",Bongi,Sem Localidade,NaN,2022-04-19,Analise Visual Estrutural,NaN,NaN,NaN,0
1,2022,ABRIL,2022-04-19,NaN,NaN,"""Avenida Dantas Barreto","`=-[""",Santo Antonio,Sem Localidade,NaN,2022-04-19,atendimento Social,NaN,NaN,NaN,0
2,2022,ABRIL,2022-04-19,NaN,NaN,"""Rua Velha","?_$.""",Boa Vista,Sem Localidade,NaN,2022-04-19,atendimento Social,NaN,NaN,NaN,0


,Regional,Ano,Mês,Data,Ocorrencia,Solicitacao,Endereco,Bairro,Localidade,Grau_de_Risco,Data_da_Acao,Tipo_da_Acao,Quantidade,Altura,Largura,Area_m
0,Oeste,2023,ABRIL,2023-04-27,NaN,NaN,"Rua Monte Pascoal, ""#*9",Cohab,Jardim Monte Verde,NaN,2023-03-27,ATENDIMENTO SOCIAL,NaN,NaN,NaN,0
1,Oeste,2023,ABRIL,2023-04-28,NaN,Monitoramento,"Rua Barra do Sagi, $!>=",Varzea,Brega e Chique,R3,2023-04-28,NaN,NaN,NaN,NaN,0
2,Oeste,2023,ABRIL,2023-04-28,NaN,Monitoramento,"Rua Jose Antonio da Costa Filho, >#'\",Varzea,Jardim Teresopolis,R3,2023-04-28,NaN,NaN,NaN,NaN,0


#### Correção do problema
A função **consertar_2022** resolve esses problemas. Primeiro, junta os dois campos do endereço com uma vírgula e remove as aspas que sobraram da quebra. Em seguida, atribui a cada coluna deslocada o valor que estava na anterior, e cria a coluna **Regional** vazia, já que essa informação não existe no arquivo de 2022.

In [4]:
def consertar_2022(df):
    # refaz o endereço, que veio quebrado em dois campos
    rua = df["Solicitacao"].fillna("").str.lstrip('"')
    numero = df["Endereco"].fillna("").str.rstrip('"')
    endereco = rua + "," + numero
    
    # realinha as colunas deslocadas
    colunas_corrigidas = pd.DataFrame({
        "Regional": pd.NA,
        "Ano": df["Regional"],
        "Mês": df["Ano"],
        "Data": df["Mês"],
        "Ocorrencia": df["Data"],
        "Solicitacao": df["Ocorrencia"],
        "Endereco": endereco
    })

    colunas_alinhadas = ["Bairro", "Localidade", "Grau_de_Risco", "Data_da_Acao", "Tipo_da_Acao", "Quantidade", "Altura", "Largura", "Area_m"]

    return pd.concat([colunas_corrigidas, df[colunas_alinhadas]], axis=1) # axis=1 concatena lado a lado

df_2022 = consertar_2022(df_2022)
df_2022.head(3)

,Regional,Ano,Mês,Data,Ocorrencia,Solicitacao,Endereco,Bairro,Localidade,Grau_de_Risco,Data_da_Acao,Tipo_da_Acao,Quantidade,Altura,Largura,Area_m
0,<NA>,2022,ABRIL,2022-04-19,Imveis com Danos / Risco,Vistoria em Imveis,"Rua Jose Moreira Reis, {})!",Bongi,Sem Localidade,NaN,2022-04-19,Analise Visual Estrutural,NaN,NaN,NaN,0
1,<NA>,2022,ABRIL,2022-04-19,NaN,NaN,"Avenida Dantas Barreto, `=-[",Santo Antonio,Sem Localidade,NaN,2022-04-19,atendimento Social,NaN,NaN,NaN,0
2,<NA>,2022,ABRIL,2022-04-19,NaN,NaN,"Rua Velha, ?_$.",Boa Vista,Sem Localidade,NaN,2022-04-19,atendimento Social,NaN,NaN,NaN,0


#### Consolidação das bases
Cada base recebe a coluna **ano_origem**, que indica de qual arquivo a linha veio. Isso é necessário porque os arquivos são organizados pela data da ação, e alguns atendimentos têm data de registro em outro ano. Depois, as três bases são empilhadas numa só. O total esperado é de 253.762 linhas.

In [5]:
df_2022["ano_origem"] = "2022"
df_2023["ano_origem"] = "2023"
df_2024["ano_origem"] = "2024"

df = pd.concat([df_2022, df_2023, df_2024], ignore_index=True)
print(f"Total de linhas da base consolidada: {len(df)}")

Total de linhas da base consolidada: 253762


#### Renomeação das colunas
Os nomes das colunas são padronizados em letras minúsculas, sem acentos e com palavras separadas por *underscore*. Os conectivos também são removidos, seguindo o mesmo padrão dos nomes das dimensões do modelo. Além disso, a coluna **Data** é renomeada para **data_registro**, para diferenciá-la da data da ação, já que cada atendimento possui as duas. Isso evita erros de digitação no código e mantém os nomes iguais em todas as etapas do projeto, incluindo as consultas em SQL.

In [6]:
df = df.rename(columns={
    "Regional": "regional",
    "Ano": "ano",
    "Mês": "mes",
    "Data": "data_registro",
    "Ocorrencia": "ocorrencia",
    "Solicitacao": "solicitacao",
    "Endereco": "endereco",
    "Bairro": "bairro",
    "Localidade": "localidade",
    "Grau_de_Risco": "grau_risco",
    "Data_da_Acao": "data_acao",
    "Tipo_da_Acao": "tipo_acao",
    "Quantidade": "quantidade",
    "Altura": "altura",
    "Largura": "largura",
    "Area_m": "area_m"    
})

df.head(3)

,regional,ano,mes,data_registro,ocorrencia,solicitacao,endereco,bairro,localidade,grau_risco,data_acao,tipo_acao,quantidade,altura,largura,area_m,ano_origem
0,<NA>,2022,ABRIL,2022-04-19,Imveis com Danos / Risco,Vistoria em Imveis,"Rua Jose Moreira Reis, {})!",Bongi,Sem Localidade,NaN,2022-04-19,Analise Visual Estrutural,NaN,NaN,NaN,0,2022
1,<NA>,2022,ABRIL,2022-04-19,NaN,NaN,"Avenida Dantas Barreto, `=-[",Santo Antonio,Sem Localidade,NaN,2022-04-19,atendimento Social,NaN,NaN,NaN,0,2022
2,<NA>,2022,ABRIL,2022-04-19,NaN,NaN,"Rua Velha, ?_$.",Boa Vista,Sem Localidade,NaN,2022-04-19,atendimento Social,NaN,NaN,NaN,0,2022


#### Tratamento das medidas numéricas

Durante o carregamento todas as medidas foram importadas como strings, inclusive as colunas originalmente numéricas.As colunas **quantidade**, **altura**, **largura** e **area_m** foram convertidas para o tipo numérico, e os valores nulos foram mantidos, já que essas medidas só se aplicam a parte dos atendimentos. Na coluna **area_m**, o valor 0 indica a ausência de medida, e não uma área nula, por isso é substituído por nulo, evitando distorções no cálculo de médias. Também foi criada a colunas de sinalização: **flag_area_divergente**, que identifica áreas diferentes do produto de altura, largura e quantidade.

In [9]:
print(df.dtypes)

regional         object
ano                 str
mes                 str
data_registro       str
ocorrencia          str
solicitacao         str
endereco            str
bairro              str
localidade          str
grau_risco          str
data_acao           str
tipo_acao           str
quantidade          str
altura              str
largura             str
area_m              str
ano_origem          str
dtype: object


In [22]:
for coluna in {"quantidade", "altura", "largura", "area_m"}:
    linhas_em_branco = df[f"{coluna}"].isna().sum()
    linhas_totais = len(df[f"{coluna}"])
    print(f'{coluna}: {linhas_em_branco} linhas em branco de {linhas_totais} no total ({(linhas_em_branco/linhas_totais)*100:.2f}%).')

largura: 220209 linhas em branco de 253762 no total (86.78%).
quantidade: 215779 linhas em branco de 253762 no total (85.03%).
altura: 220193 linhas em branco de 253762 no total (86.77%).
area_m: 0 linhas em branco de 253762 no total (0.00%).


In [ ]:
MEDIDAS = ["quantidade", "altura", "largura", "area_m"]

def tratar_medidas(df):
    df = df.copy()

    for col in MEDIDAS:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # tratamento das áreas nulas
    df["area_m"] = df["area_m"].where(df["area_m"] > 0)

    # tratamento das áreas calculadas erradas
    esperado = df["altura"] * df["largura"] * df["quantidade"]
    df["flag_area_divergente"] = (
        (esperado.notna() & ~((df["area_m"] - esperado).abs() <= 0.01))
        | (df["area_m"].notna() & esperado.isna())
    )
    return df

df = tratar_medidas(df)
assert len(df) == 253762

In [29]:
print(df.dtypes)

regional                 object
ano                         str
mes                         str
data_registro               str
ocorrencia                  str
solicitacao                 str
endereco                    str
bairro                      str
localidade                  str
grau_risco                  str
data_acao                   str
tipo_acao                   str
quantidade              float64
altura                  float64
largura                 float64
area_m                  float64
ano_origem                  str
flag_area_divergente       bool
dtype: object


#### Tratamento de linhas duplicadas

A base não possui um identificador único de atendimento, então não é possível saber diretamente se linhas idênticas são registros repetidos por engano ou atendimentos distintos com as mesmas características. Para avaliar o problema, são contadas as linhas totalmente iguais às anteriores dentro da base consolidada, por ano de origem.

In [34]:
colunas_base = [c for c in df.columns if c != "flag_area_divergente"]
dup = df.duplicated(subset=colunas_base, keep="first")

resumo = dup.groupby(df["ano_origem"]).agg(duplicatas="sum", total="size")
resumo["percentual"] = (resumo["duplicatas"] / resumo["total"] * 100).round(2)
resumo

,duplicatas,total,percentual
ano_origem,,,
2022,5735,90425,6.34
2023,5897,83164,7.09
2024,5931,80173,7.40


In [36]:
def remover_duplicatas(df):
    colunas_base = [c for c in df.columns if c != "flag_area_divergente"]
    return df.drop_duplicates(subset=colunas_base, keep="first").reset_index(drop=True)

antes = len(df)
df = remover_duplicatas(df)
print(f"{antes - len(df)} linhas removidas; {len(df)} restantes")

17563 linhas removidas; 236199 restantes


Foram identificadas 17.563 linhas duplicadas, correspondentes a 6,9% da base, distribuídas de forma semelhante entre os três anos (entre 6,3% e 7,4%). As duplicatas ocorrem apenas dentro de um mesmo arquivo e se concentram em ações porta a porta, que representam 53% delas, contra 16% da base. Como as linhas são idênticas em todas as colunas, inclusive data e endereço, assume-se que se tratam de registros repetidos, e as cópias são removidas, mantendo-se a primeira ocorrência.

### Carregamento dos dados

*Esta seção será preenchida após a construção da tabela fato e das dimensões.*